# Mastering Python Concurrency: Free-threading & Sub-interpreters

This notebook synthesizes all six course modules into a single, runnable reference. Every major concept is explained and followed by working code you can execute and experiment with.

---
## Table of Contents

1. [The GIL — Why Threads Don't Scale](#module-1)
2. [Mixing asyncio with CPU-bound Work](#module-2)
3. [Running Python Without the GIL (Free-threading)](#module-3)
4. [Isolating Parallel Work with Sub-interpreters](#module-4)
5. [Profiling Free-threaded and Sub-interpreter Workloads](#module-5)
6. [Shipping Free-threaded Python to Production](#module-6)
7. [Decision Guide: Choosing Your Strategy](#decision-guide)

---

## Running Example: Globomantics Risk Engine

Throughout this course, every benchmark uses the **Globomantics Risk Engine** — a CPU-bound Monte Carlo portfolio-risk simulator that runs thousands of price scenarios to produce a `var95` (Value at Risk, 95th percentile) figure. It is representative of real scientific/financial workloads: pure Python computation with negligible I/O.

Key benchmark invariant: **40,000 scenarios** are pinned across the entire course so all strategy comparisons are honest.

---
<a id='module-1'></a>
## Module 1 — The GIL: Why Threads Don't Scale

### What is the GIL?

The **Global Interpreter Lock (GIL)** is a process-wide mutex inside CPython. A thread must hold it before executing any Python bytecode. Only one thread holds the GIL at any moment, so threads take *turns* — concurrency, never parallelism.

```
Thread A: ██████░░░░░░██████░░░░░░
Thread B: ░░░░░░██████░░░░░░██████
              ↑ 5ms switch interval
```

The interpreter yields the GIL every ~5 ms (`sys.getswitchinterval()`). Threads *appear* concurrent but they never actually run Python bytecode in parallel.

### Why does CPython need a GIL?

Every Python object carries a **reference count** — an integer that tracks how many names point to it. When the count hits zero the object is freed. Updating that counter is **not atomic**; without protection two threads could corrupt it simultaneously, causing use-after-free bugs or memory leaks.

The GIL is the simplest solution: hold the lock → only one thread can change any ref count at a time.

### Python Concurrency Timeline

| Year | Feature | Notes |
|------|---------|-------|
| Early 1990s | `threading` | GIL-limited concurrency |
| Python 2.6 | `multiprocessing` | Real parallelism via processes |
| Python 3.2 | `concurrent.futures` | High-level thread/process pools |
| Python 3.4 | `asyncio` | Single-thread cooperative I/O |
| Python 3.12 | PEP 684 | Per-interpreter GIL (foundation) |
| Python 3.13 | PEP 703 | Free-threading experimental |
| Python 3.14 | PEP 779 | Free-threading officially supported |
| Python 3.14 | PEP 734 | Sub-interpreters in stdlib |

In [1]:
import sys
import threading
import time

print(f"Python version: {sys.version}")
print(f"GIL switch interval: {sys.getswitchinterval()*1000:.1f} ms")
print(f"GIL enabled: {getattr(sys, '_is_gil_enabled', lambda: True)()}")

Python version: 3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 21.0.0 (clang-2100.0.123.102)]
GIL switch interval: 5.0 ms
GIL enabled: True


### Demonstrating the GIL bottleneck

On CPU-bound work, adding threads gives **no speedup** — each thread holds the GIL while running, so others wait.

In [2]:
import time
import threading

def cpu_work(n=5_000_000):
    """Pure-Python CPU burn — held the GIL the entire time."""
    total = 0
    for i in range(n):
        total += i * i
    return total

# ── Baseline: single thread ──────────────────────────────────────────────────
t0 = time.perf_counter()
cpu_work()
single = time.perf_counter() - t0
print(f"Single thread:  {single:.3f}s")

# ── Two threads: should be ~2× faster IF parallelism existed ─────────────────
t0 = time.perf_counter()
threads = [threading.Thread(target=cpu_work) for _ in range(2)]
for th in threads: th.start()
for th in threads: th.join()
threaded = time.perf_counter() - t0
print(f"Two threads:    {threaded:.3f}s")
print(f"Speedup:        {single/threaded:.2f}x  (expected ~2x, actual ~1x — GIL!)")

Single thread:  0.113s
Two threads:    0.198s
Speedup:        0.57x  (expected ~2x, actual ~1x — GIL!)


### IO-bound vs CPU-bound under the GIL

The GIL is **released** whenever a thread blocks on I/O (file reads, network, `time.sleep`). This is why threads still help for I/O-bound work.

| Work type | Thread behavior | GIL impact |
|-----------|----------------|------------|
| CPU-bound | Holds GIL entire run | No parallel speedup |
| IO-bound | Releases GIL on every blocking call | Good throughput |
| C extension (e.g. NumPy) | Extension can release GIL around native work | Can parallelize |

### Process-based parallelism (multiprocessing)

The classic workaround: spawn separate processes, each with its own GIL and memory space. Genuine parallelism, but with costs:

- **Spawn startup**: each new process re-imports your code (~0.3s overhead)
- **Pickling**: every argument and return value must be serialized across the boundary
- **Memory**: each worker has its own heap — no sharing
- **Amdahl ceiling**: fan-out + merge + sort stay serial, so 8 workers → ~2.9×, not 8×

In [3]:
import concurrent.futures
import time

# simulate_scenario lives in _nb_workers.py so spawn-based workers can pickle it.
# Functions defined here in __main__ cannot be unpickled in spawned processes.
from _nb_workers import simulate_scenario

N_SCENARIOS = 400  # small for demo; real engine uses 40,000

# Single process
t0 = time.perf_counter()
results_single = [simulate_scenario(i) for i in range(N_SCENARIOS)]
single = time.perf_counter() - t0
print(f"Single process: {single:.3f}s")

# Process pool
t0 = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as pool:
    results_pool = list(pool.map(simulate_scenario, range(N_SCENARIOS)))
pool_time = time.perf_counter() - t0
print(f"4-process pool: {pool_time:.3f}s")
print(f"Speedup:        {single/pool_time:.2f}x (includes spawn + pickle overhead)")

Single process: 0.420s
4-process pool: 0.197s
Speedup:        2.13x (includes spawn + pickle overhead)


---
<a id='module-2'></a>
## Module 2 — Mixing asyncio with CPU-bound Work

### The asyncio execution model

asyncio uses **cooperative multitasking** on a single thread:

```
Event Loop
    │
    ├─ coroutine A runs until it hits `await` → paused
    ├─ coroutine B runs until it hits `await` → paused  
    ├─ coroutine C runs until it hits `await` → paused
    └─ (OS signals readiness → resume whoever is waiting)
```

**Key rules:**
- A coroutine runs until it explicitly `await`s — there is no 5ms preemption timer
- A coroutine that never awaits never yields → it **starves the loop**
- The event loop owns the sockets; all I/O multiplexing happens there

### Event-loop starvation

If you run CPU work directly inside a coroutine, the entire event loop — and all other coroutines — freeze until it completes. Rule of thumb: **more than a few milliseconds of CPU = not in a coroutine**.

In [4]:
import asyncio
import time

async def fast_ping():
    """Simulates a quick heartbeat check."""
    await asyncio.sleep(0)
    return "pong"

async def cpu_hog():
    """CPU work inline on the event loop — WRONG pattern."""
    total = 0
    for i in range(3_000_000):   # ~100ms of work
        total += i
    return total

async def demo_starvation():
    t0 = time.perf_counter()
    # Start a ping that should return almost instantly
    ping_task = asyncio.create_task(fast_ping())
    # But hog the loop first
    await cpu_hog()
    ping_latency = time.perf_counter() - t0
    await ping_task
    print(f"ping() returned after {ping_latency*1000:.0f}ms — blocked by CPU hog")

# Jupyter already runs an event loop, so use `await` directly instead of asyncio.run()
await demo_starvation()

ping() returned after 40ms — blocked by CPU hog


### The correct hybrid architecture

Keep asyncio as the I/O front door; push CPU work into a thread pool. The handoff point is `asyncio.to_thread()`.

```
Async event loop  ──── await asyncio.to_thread(run_job, args) ────▶  Thread pool
      │                        │                                          │
   (I/O, routing)          (handoff)                              (CPU simulation)
```

The event loop is **never blocked**: it awaits a future that resolves when the thread finishes.

In [5]:
import asyncio
import time
import random

def run_simulation(n_scenarios, seed=42):
    """CPU-bound work — safe to run in a thread pool worker."""
    rng = random.Random(seed)
    results = [sum(rng.gauss(0, 1) for _ in range(1000)) for _ in range(n_scenarios)]
    results.sort()
    var95 = results[int(len(results) * 0.05)]  # 5th percentile loss
    return var95

async def handle_request(job_id: int):
    """Async handler — offloads CPU work without blocking the loop."""
    print(f"  Job {job_id}: starting")
    # asyncio.to_thread() is the explicit handoff point
    result = await asyncio.to_thread(run_simulation, n_scenarios=200, seed=job_id)
    print(f"  Job {job_id}: done  var95={result:.2f}")
    return result

async def main():
    print("Firing 5 concurrent risk jobs...")
    t0 = time.perf_counter()
    results = await asyncio.gather(*[handle_request(i) for i in range(5)])
    elapsed = time.perf_counter() - t0
    print(f"\nAll done in {elapsed:.3f}s  (jobs ran concurrently in thread pool)")

await main()

Firing 5 concurrent risk jobs...
  Job 0: starting
  Job 1: starting
  Job 2: starting
  Job 3: starting
  Job 4: starting
  Job 1: done  var95=-46.84
  Job 0: done  var95=-52.84
  Job 3: done  var95=-53.59
  Job 2: done  var95=-44.67
  Job 4: done  var95=-47.46

All done in 0.107s  (jobs ran concurrently in thread pool)


### Hybrid cost analysis

| Component | Cost | Notes |
|-----------|------|-------|
| Thread creation | ~50µs | Cheap compared to process spawn |
| Thread switch | ~1–5µs | OS-level context switch |
| `asyncio.to_thread` overhead | ~10µs | Future wrapper cost |
| Total handoff cost | Negligible vs CPU work | Fine for >1ms jobs |

The hybrid approach gives you:
- asyncio's I/O efficiency (one thread handles thousands of connections)
- Thread pool's CPU parallelism (GIL released between Python calls)
- No spawn overhead (thread pool workers persist)

---
<a id='module-3'></a>
## Module 3 — Running Python Without the GIL (Free-threading)

### What is free-threading?

**PEP 703** (experimental in Python 3.13) and **PEP 779** (officially supported in Python 3.14) remove the GIL. Threads can now execute Python bytecode **truly in parallel**.

Two binaries ship:
- `python3.14` — standard build (GIL present, same as always)
- `python3.14t` — free-threaded build (no GIL)

### Installing the free-threaded build

```bash
# Using uv (recommended)
uv python install 3.14t
uv run --python 3.14t your_script.py

# Check at runtime
python3.14t -c "import sys; print(sys._is_gil_enabled())"
# → False
```

### How free-threading solves the ref-count problem

Without the GIL, concurrent ref-count updates would corrupt memory. PEP 703 uses three techniques:

#### 1. Per-object locking
Each mutable object carries its own lightweight lock. Mutation of object A doesn't block mutation of object B.

#### 2. Biased reference counting
Each object has **two counters**: a local (thread-local, no atomics needed) and a shared (atomic). The creating thread updates the local counter cheaply; other threads use the atomic shared counter. Most objects are only touched by one thread → near-zero overhead.

#### 3. Deferred reference counting + immortal objects
Modules, classes, and functions skip eager ref-count updates; the garbage collector accounts for them at collection time. `None`, `True`, `False`, and small integers are **immortal** — their ref count never changes at all.

In [6]:
import sys

# Check whether the GIL is active in the current interpreter
if hasattr(sys, '_is_gil_enabled'):
    gil_status = sys._is_gil_enabled()
    print(f"GIL enabled: {gil_status}")
    if not gil_status:
        print("Running on free-threaded build — true thread parallelism available!")
    else:
        print("Running on standard build — threads take turns.")
else:
    print("Python < 3.13: GIL always active (sys._is_gil_enabled not available)")

GIL enabled: True
Running on standard build — threads take turns.


### Single-thread overhead

The free-threaded build trades a ~20% single-thread slowdown for the ability to scale across cores:

| Metric | Standard build | Free-threaded build |
|--------|---------------|--------------------|
| Single thread | ~1.41s | ~1.70s (−20%) |
| 8 threads | ~1.41s (GIL → no gain) | ~0.46s (+3.7×) |
| Net at 8 workers | 1× baseline | **~3.7× faster** |

The question is whether your workload is single-threaded or multi-threaded in production.

### Data races: the hidden danger

**The GIL never promised thread-safety for your data.** It only protected the interpreter's own internals. But the GIL's 5ms scheduling granularity accidentally masked many races — threads rarely interleaved inside your code. Free-threading removes that accidental protection.

**Thread-safety red zones:**

In [7]:
import threading

# ── Red Zone 1: Shared mutable global ────────────────────────────────────────
# BAD — classic data race
counter = 0

def increment_racy():
    global counter
    for _ in range(100_000):
        counter += 1  # read-modify-write is NOT atomic!

# On free-threaded build this will produce a value < 200,000
threads = [threading.Thread(target=increment_racy) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Racy counter: {counter}  (expected 200000, got less on free-threaded build)")

# ── Red Zone 1 (fixed): Use a Lock ───────────────────────────────────────────
counter_safe = 0
lock = threading.Lock()

def increment_safe():
    global counter_safe
    for _ in range(100_000):
        with lock:
            counter_safe += 1

threads = [threading.Thread(target=increment_safe) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Safe counter:  {counter_safe}  (always exactly 200000)")

Racy counter: 200000  (expected 200000, got less on free-threaded build)
Safe counter:  200000  (always exactly 200000)


In [8]:
import threading

# ── Red Zone 2: Lazy initialization / check-then-act ─────────────────────────
_cache = {}

def get_value_racy(key):
    # WRONG: two threads can both see cache miss and both compute
    if key not in _cache:
        _cache[key] = expensive_compute(key)  # double-compute race
    return _cache[key]

# CORRECT pattern using a lock
_cache_lock = threading.Lock()

def get_value_safe(key):
    with _cache_lock:
        if key not in _cache:
            _cache[key] = expensive_compute(key)
        return _cache[key]

def expensive_compute(k):
    return k * k

# Or use functools.lru_cache — it is thread-safe
import functools

@functools.lru_cache(maxsize=None)
def get_value_cached(key):
    return expensive_compute(key)

print("get_value_cached(5):", get_value_cached(5))
print("Cache is thread-safe via lru_cache")

get_value_cached(5): 25
Cache is thread-safe via lru_cache


In [9]:
import threading

# ── Red Zone 3: Invariants spanning multiple fields ───────────────────────────
class Portfolio:
    def __init__(self):
        self.cash = 1000.0
        self.shares = 0
        self._lock = threading.Lock()  # protects both fields together

    def buy_racy(self, price, qty):
        # BAD: between the two writes, another thread sees inconsistent state
        self.cash -= price * qty
        self.shares += qty          # window of inconsistency here!

    def buy_safe(self, price, qty):
        with self._lock:            # both updates happen atomically
            self.cash -= price * qty
            self.shares += qty

p = Portfolio()
p.buy_safe(10.0, 5)
print(f"After buy: cash={p.cash}, shares={p.shares}")

After buy: cash=950.0, shares=5


### Race detection tools

1. **ThreadSanitizer (TSan)**: Build the interpreter with `-fsanitize=thread` — catches data races at runtime with ~5–10× overhead. Only for CI, not production.
2. **Invariant checking in production**: Add assertions that verify your data model is consistent. Cheap to check, fast to catch races.
3. **Dual-build CI matrix**: Run your entire test suite on *both* `python3.14` and `python3.14t`. Failures on `t` only → race condition.

### Free-threading adoption checklist

| Area | What to check | Action |
|------|--------------|--------|
| **C extensions** | Does every dependency have a `cp314t` wheel? | Check PyPI, `Py_mod_gil` slot |
| **Global state** | Any shared mutable module-level variables? | Add locks or use thread-local storage |
| **Test obligations** | Are tests passing on both builds? | Add `python3.14t` to CI matrix |
| **Benchmarking** | Did you measure with the right denominator? | Benchmark against the build you ship |

> **Critical**: One incompatible C extension import **re-enables the GIL process-wide** — only a warning is emitted. Use `sys._is_gil_enabled()` at startup to verify the GIL is actually off.

In [10]:
import sys

def assert_free_threaded():
    """Wire this check into service startup when you need the GIL off."""
    if hasattr(sys, '_is_gil_enabled') and sys._is_gil_enabled():
        raise RuntimeError(
            "GIL is active! A C extension may have re-enabled it.\n"
            "Run with: python3.14t -X gil=0 your_service.py"
        )

try:
    assert_free_threaded()
    print("GIL is off — free-threaded mode confirmed")
except RuntimeError as e:
    print(f"Startup check failed: {e}")

Startup check failed: GIL is active! A C extension may have re-enabled it.
Run with: python3.14t -X gil=0 your_service.py


---
<a id='module-4'></a>
## Module 4 — Isolating Parallel Work with Sub-interpreters

### Two safety models for parallel Python

| | Free-threading | Sub-interpreters |
|---|---|---|
| **State** | Threads share everything by default | Share nothing by default |
| **Safety** | Discipline: locks, partials, stress tests | Construction: races are structurally impossible |
| **GIL** | No GIL (free-threaded build only) | One GIL per interpreter |
| **Build** | Requires `python3.14t` | Works on standard `python3.14` |

### The sub-interpreter model

A sub-interpreter is a **complete, independent Python world inside one process**. It has its own:
- Module registry (`sys.modules`)
- Global variables
- `sys.path`
- **GIL** (since PEP 684, Python 3.12)

Since interpreters cannot see each other's objects, **races require two threads mutating one object — which is structurally impossible**. Safety by construction, not discipline.

### PEP timeline

- **PEP 684 (Python 3.12)**: Each interpreter gets a private GIL
- **PEP 734 (Python 3.14)**: Creation and control via the standard library (`concurrent.interpreters`)

### Comparison: Threads vs Sub-interpreters vs Processes

| | Threads | Sub-interpreters | Processes |
|---|---|---|---|
| Memory | One shared heap | Process-like isolation | Full isolation |
| Cost | Cheapest | Close to thread cost | Spawn + full interpreter |
| Risk | Everything shared, highest | Nothing shared by accident | No shared state |
| IPC | Direct object ref | Queues / shareable types | Pickle on every crossing |

### Using the API

Two levels in Python 3.14:

```python
# High-level: concurrent.futures executor (familiar interface)
from concurrent.futures import InterpreterPoolExecutor

with InterpreterPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(simulate_scenario, seed) for seed in range(N)]
    results = [f.result() for f in futures]

# Low-level: create interpreters directly, wire queues by hand
import concurrent.interpreters
interp = concurrent.interpreters.create()
```

In [11]:
# Demonstration of InterpreterPoolExecutor (Python 3.14+)
# Falls back gracefully on earlier versions

import sys
import time

def simulate_portfolio(seed):
    """Worker function — must be importable (not defined in __main__)."""
    import random
    rng = random.Random(seed)
    returns = [rng.gauss(0.0005, 0.02) for _ in range(252)]  # one trading year
    sorted_returns = sorted(returns)
    var95 = sorted_returns[int(len(sorted_returns) * 0.05)]
    return round(var95, 6)

N = 100  # number of portfolios to simulate

if sys.version_info >= (3, 14):
    from concurrent.futures import InterpreterPoolExecutor
    print("Using InterpreterPoolExecutor (Python 3.14)")
    t0 = time.perf_counter()
    with InterpreterPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(simulate_portfolio, range(N)))
    elapsed = time.perf_counter() - t0
    print(f"Sub-interpreters: {N} portfolios in {elapsed:.3f}s")
    print(f"Sample var95 values: {results[:5]}")
else:
    # Fallback: demonstrate with process pool (same API shape)
    from concurrent.futures import ProcessPoolExecutor
    print(f"Python {sys.version_info.major}.{sys.version_info.minor}: "
          f"InterpreterPoolExecutor requires Python 3.14+")
    print("Demonstrating with ProcessPoolExecutor (same interface):")
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(simulate_portfolio, range(N)))
    elapsed = time.perf_counter() - t0
    print(f"Process pool: {N} portfolios in {elapsed:.3f}s")
    print(f"Sample var95 values: {results[:5]}")

Using InterpreterPoolExecutor (Python 3.14)
Sub-interpreters: 100 portfolios in 0.053s
Sample var95 values: [-0.03511, -0.028534, -0.03216, -0.03772, -0.02977]


### The costs of isolation

Sub-interpreters are not free. Each worker pays:

- **Per-worker imports**: each interpreter imports its own copy of your modules (startup time + memory)
- **Crossing cost**: most values must be **pickled** across the boundary
- **Shareable types**: a small set crosses directly — `int`, `float`, `bytes`, `str`, `None`, `memoryview`

This is "a milder multiprocessing tax, without the processes."

### How the callable crosses the boundary

The pool pickles the callable by **reference** — module path plus name. Each worker interpreter re-imports the module and looks up the name. This means:

- ✅ Functions defined in a proper module work fine
- ❌ Functions defined in `__main__` fail (the sub-interpreter's `__main__` is blank)
- ❌ Lambdas and closures that touch globals fail at crossing time

**Contrast with multiprocessing spawn**: multiprocessing re-runs your entry script in the child, so `__main__`-defined functions usually survive. Sub-interpreters never re-run your entry script.

### Performance: ~3× at 8 workers on the standard build

Both free-threading and sub-interpreters deliver **~3× speedup at 8 workers** on the risk engine. The same Amdahl ceiling applies — fan-out, merge, and sort stay serial.

---
<a id='module-5'></a>
## Module 5 — Profiling Free-threaded and Sub-interpreter Workloads

### Why measurement is hard

Every "3×" speedup claim in the course is a **wall-clock reading** — not a guarantee. Honest benchmarking requires understanding the sources of measurement error.

### Sources of measurement error

| Source | Description | Fix |
|--------|-------------|-----|
| **Warm-up** | First runs pay import, allocator, and cache-fill costs | Warm up first, then time |
| **Variance** | OS schedules other work around your benchmark | Report best-of-N or distribution |
| **Interference** | Some instruments change what they measure | Use sampling profiler for sizing, plain timing for perf numbers |

### Amdahl's Law

Even perfectly parallelizable work has a serial fraction. Amdahl's Law tells you the ceiling:

```
speedup = 1 / ((1 - p) + p/N)

where:
  p = fraction of the run that parallelizes
  N = number of workers
```

The risk engine's scenario loop parallelizes almost perfectly. But fan-out, merging, and sorting 40k P&Ls stay serial — which is part of why 8 workers deliver ~3×, not 8×.

In [12]:
def amdahl_speedup(parallel_fraction: float, n_workers: int) -> float:
    """Calculate theoretical max speedup given Amdahl's Law."""
    serial_fraction = 1 - parallel_fraction
    return 1 / (serial_fraction + parallel_fraction / n_workers)

print("Amdahl's Law — theoretical speedup ceilings:")
print(f"{'Workers':>8} | {'p=0.80':>8} | {'p=0.90':>8} | {'p=0.95':>8} | {'p=0.99':>8}")
print("-" * 52)
for n in [2, 4, 8, 16, 32]:
    row = [amdahl_speedup(p, n) for p in [0.80, 0.90, 0.95, 0.99]]
    print(f"{n:>8} | {row[0]:>8.2f} | {row[1]:>8.2f} | {row[2]:>8.2f} | {row[3]:>8.2f}")

print()
print("Risk engine: p≈0.90 → 8 workers → ceiling ~4.7x (measured ~3x due to overhead)")

Amdahl's Law — theoretical speedup ceilings:
 Workers |   p=0.80 |   p=0.90 |   p=0.95 |   p=0.99
----------------------------------------------------
       2 |     1.67 |     1.82 |     1.90 |     1.98
       4 |     2.50 |     3.08 |     3.48 |     3.88
       8 |     3.33 |     4.71 |     5.93 |     7.48
      16 |     4.00 |     6.40 |     9.14 |    13.91
      32 |     4.44 |     7.80 |    12.55 |    24.43

Risk engine: p≈0.90 → 8 workers → ceiling ~4.7x (measured ~3x due to overhead)


### Three metrics for a concurrent service

| Metric | How to measure | What to watch |
|--------|---------------|---------------|
| **Throughput** | Jobs/second at real concurrency | Client prints jobs/s |
| **Latency tail** | p95 and p99, not the mean | One ping blocked 7+ sec: invisible in mean, glaring at p99 |
| **Cores-used** | `process_time() / perf_counter()` | Separates parallel from "taking turns" |

Wall-clock alone cannot tell parallel from lucky. A GIL-bound run where threads take turns looks the same as a free-threaded run if you only read the wall clock — but `cores-used` exposes the difference.

In [13]:
import time
import threading
import concurrent.futures

def cpu_work_timed(n=1_000_000):
    total = 0
    for i in range(n):
        total += i
    return total

def measure_parallelism(n_workers=4, work_per_worker=500_000):
    """Measure wall-clock and CPU-time to detect real parallelism."""
    t_wall_start = time.perf_counter()
    t_cpu_start = time.process_time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = [pool.submit(cpu_work_timed, work_per_worker) for _ in range(n_workers)]
        results = [f.result() for f in futures]

    wall = time.perf_counter() - t_wall_start
    cpu = time.process_time() - t_cpu_start
    cores_used = cpu / wall

    print(f"Workers: {n_workers}")
    print(f"  Wall clock:  {wall:.3f}s")
    print(f"  CPU time:    {cpu:.3f}s")
    print(f"  Cores used:  {cores_used:.2f}")
    if cores_used < 1.1:
        print("  → Threads taking turns (GIL active or IO-bound)")
    else:
        print(f"  → True parallelism! Using {cores_used:.1f} cores")

measure_parallelism(n_workers=4)

Workers: 4
  Wall clock:  0.027s
  CPU time:    0.028s
  Cores used:  1.04
  → Threads taking turns (GIL active or IO-bound)


### The 5 benchmark traps

**Trap 1 — Single-thread overhead**: Free-threaded build is ~20% slower on a single thread. If you benchmark free-threaded `python3.14t` against standard `python3.14` and only show the parallel case, you're hiding the overhead. *Always report both denominators.*

**Trap 2 — Unrepresentative inputs**: Tiny workloads let spawn/import/startup costs dominate. The process pool "won" at tiny input sizes simply because 40k scenarios amortized its 0.3s spawn cost. *Pin a fixed, representative workload.*

**Trap 3 — Skipping warm-up**: First runs pay import cost, allocator growth, and cold caches. One cold reading can disagree wildly with warmed repeats. *Warm up, then time.*

**Trap 4 — Believing one number**: Wall-clock alone can't distinguish parallel from lucky. Pair every wall reading with `cores-used`; for live services, add the latency tail.

**Trap 5 — Measuring through the profiler**: `cProfile` ranked the hotspot correctly but inflated its seconds by 3.7×. Use deterministic profiling (cProfile) to rank, sampling profiler (py-spy, Austin) or plain `time.perf_counter()` to size. *Never quote profiled seconds as performance.*

### Comparing the four strategies

Because the GIL is fixed at interpreter startup, you can't run all four strategies in one process. Use **two binaries in CI**, same code:

```bash
# Strategy A/B/C on standard build:
python3.14 compare.py --strategy serial
python3.14 compare.py --strategy threads    # → GIL threads (IO-bound only)
python3.14 compare.py --strategy processes
python3.14 compare.py --strategy interpreters

# Strategy D on free-threaded build:
python3.14t compare.py --strategy threads   # → free threads (CPU-bound)
```

### The honest benchmarking method

1. Fix a representative workload (40,000 scenarios)
2. Warm up, then take best-of-N
3. Report wall, cores-used, and throughput per strategy — same table every time
4. In production: put measurement behind a live metrics endpoint
5. See the speedup, then defend it

In [14]:
import time
import concurrent.futures

# benchmark_workload lives in _nb_workers.py for the same reason as simulate_scenario:
# ProcessPoolExecutor uses spawn and can only pickle importable callables.
from _nb_workers import benchmark_workload


def benchmark(strategy_name, executor_cls, max_workers, n_jobs=200, n_repeats=3):
    """Run a strategy N times and report wall, CPU, cores-used."""
    walls = []
    cpus = []

    # Warm up
    with executor_cls(max_workers=max_workers) as pool:
        list(pool.map(benchmark_workload, range(10)))

    for _ in range(n_repeats):
        t_wall = time.perf_counter()
        t_cpu = time.process_time()
        with executor_cls(max_workers=max_workers) as pool:
            list(pool.map(benchmark_workload, range(n_jobs)))
        walls.append(time.perf_counter() - t_wall)
        cpus.append(time.process_time() - t_cpu)

    best_wall = min(walls)
    best_cpu = min(cpus)
    cores = best_cpu / best_wall
    throughput = n_jobs / best_wall
    print(f"{strategy_name:<22} wall={best_wall:.3f}s  "
          f"cores={cores:.2f}  throughput={throughput:.0f} jobs/s")


print("Strategy comparison (same 200-job workload, best-of-3):")
print("-" * 70)
benchmark("1 thread (serial)",    concurrent.futures.ThreadPoolExecutor,   max_workers=1)
benchmark("4 threads (GIL)",      concurrent.futures.ThreadPoolExecutor,   max_workers=4)
benchmark("4 processes",          concurrent.futures.ProcessPoolExecutor,  max_workers=4)
print()
print("Note: 'cores' close to 1.0 means GIL threads taking turns.")
print("      'cores' > 2.0 means real parallelism (processes, or free-threaded build).")

Strategy comparison (same 200-job workload, best-of-3):
----------------------------------------------------------------------
1 thread (serial)      wall=0.043s  cores=1.00  throughput=4624 jobs/s
4 threads (GIL)        wall=0.043s  cores=1.01  throughput=4610 jobs/s
4 processes            wall=0.066s  cores=0.40  throughput=3040 jobs/s

Note: 'cores' close to 1.0 means GIL threads taking turns.
      'cores' > 2.0 means real parallelism (processes, or free-threaded build).


---
<a id='module-6'></a>
## Module 6 — Shipping Free-threaded Python to Production

### The production architecture

```
┌──────────────────────────────────────────────────────┐
│  async ingress  (event loop owns the sockets)         │
│        │                                              │
│  bounded queue  (admission control — says no early)   │
│        │                                              │
│  worker pool    (--pool threads  OR  --pool interps)  │
│        │                                              │
│  results store  (plain dict, mutated on event loop)   │
└──────────────────────────────────────────────────────┘
```

The `--pool` flag is the only difference between the two deployments — same architecture, chosen per environment.

### One server, two worker pools

| | Free-threaded threads | Sub-interpreters |
|---|---|---|
| Flag | `--pool threads` on `python3.14t` | `--pool interpreters` on `python3.14` |
| Memory | One shared heap, fastest handoffs | Per-worker imports, per-worker memory |
| Best for | Hot, trusted payloads | Untrusted, stateful, or fault-prone jobs |

### Back-pressure

An **unbounded queue** never says no — it just grows. Overload becomes silent memory growth and ballooning latency. A **bounded queue** returns `503 Service Unavailable` immediately when full. The client gets a fast failure it can retry with intent.

In [15]:
import asyncio
import time
import random
from typing import Optional

# Simplified production server pattern

class RiskEngineServer:
    """
    Async ingress + bounded queue + thread pool workers.
    Results stored in a dict mutated only on the event loop.
    """
    def __init__(self, max_queue: int = 5, n_workers: int = 3):
        self._queue: asyncio.Queue = asyncio.Queue(maxsize=max_queue)
        self._results: dict = {}  # mutated only on the event loop
        self._n_workers = n_workers

    def _run_simulation(self, job_id: int, n_scenarios: int) -> float:
        """CPU work — runs in thread pool, never blocks the event loop."""
        rng = random.Random(job_id)
        results = sorted(sum(rng.gauss(0, 1) for _ in range(100))
                         for _ in range(n_scenarios))
        return results[int(len(results) * 0.05)]  # var95

    async def _worker(self):
        while True:
            job_id, n_scenarios = await self._queue.get()
            try:
                result = await asyncio.to_thread(
                    self._run_simulation, job_id, n_scenarios
                )
                self._results[job_id] = result  # safe: event-loop only
            finally:
                self._queue.task_done()

    async def submit(self, job_id: int, n_scenarios: int = 50) -> Optional[str]:
        """Returns None if accepted, '503' if queue full (back-pressure)."""
        try:
            self._queue.put_nowait((job_id, n_scenarios))
            return None  # accepted
        except asyncio.QueueFull:
            return "503"  # shed immediately

    async def run_demo(self, n_jobs: int = 12):
        workers = [asyncio.create_task(self._worker()) for _ in range(self._n_workers)]
        accepted = rejected = 0
        for i in range(n_jobs):
            status = await self.submit(i)
            if status == "503":
                print(f"  Job {i:2d}: 503 — queue full, shedding")
                rejected += 1
            else:
                accepted += 1
            await asyncio.sleep(0.01)  # simulate request arrival spacing

        await self._queue.join()  # drain in-flight work
        for w in workers:
            w.cancel()

        print(f"\nAccepted: {accepted}  Rejected (503): {rejected}")
        print(f"Results computed: {len(self._results)}")
        print("Sample var95 values:", {k: round(v, 3) for k, v in list(self._results.items())[:4]})

server = RiskEngineServer(max_queue=5, n_workers=3)
await server.run_demo(n_jobs=12)


Accepted: 12  Rejected (503): 0
Results computed: 12
Sample var95 values: {0: -15.159, 1: -18.073, 2: -20.053, 3: -16.539}


### Graceful degradation

Bounded queue + fixed pool gives a **stated capacity**:
- Queue absorbs bursts (smooths spikes)
- Pool sets sustained throughput ceiling
- Overload sheds cleanly at the limit, no collapse
- Expose queue depth and in-flight count as metrics

### Failure containment

| Pattern | Benefit |
|---------|--------|
| Per-job timeouts | Protect caller's latency budget |
| Crashing job dies inside its worker | Sub-interpreters: isolation faults surface as catchable exceptions |
| Graceful shutdown | Drain in-flight work instead of dropping it |
| Chaos testing | Wire it end-to-end, then break it on purpose |

### The 4-gate migration and rollout strategy

**Gate 1 — C-extension audit**
- Check every native dependency for `cp314t` wheels on PyPI
- If not found, look for `Py_mod_gil` slot in extension code
- Wire `sys._is_gil_enabled()` check into service startup

**Gate 2 — Dual-build CI**
- Run entire test suite on both `python3.14` and `python3.14t`
- Add a stress lane with `--fail-fast` on both
- Green on standard build alone proves nothing

**Gate 3 — Feature-flagged free-threading**
- Use `-X gil=1` / `PYTHON_GIL=1` to re-enable GIL on the free-threaded binary
- One artifact, runtime picked per environment → instant rollback, no rebuild
- Canary a slice, judge tail latency and cores-used

**Gate 4 — Benchmarking gates in CI**
- Run the comparison discipline as a CI job (fixed workload, warm-up, report table)
- A regression blocks the rollout like a failing test

In [16]:
# Gate 1: C-extension audit helper
import importlib
import sys

def audit_extension(module_name: str) -> dict:
    """Check if a module is free-thread compatible."""
    try:
        mod = importlib.import_module(module_name)
        # Pure Python modules are always compatible
        is_c_ext = hasattr(mod, '__spec__') and (
            mod.__spec__ is not None and 
            mod.__spec__.origin is not None and
            '.so' in str(mod.__spec__.origin)
        )
        return {
            "module": module_name,
            "importable": True,
            "is_c_extension": is_c_ext,
            "note": "check for cp314t wheel on PyPI" if is_c_ext else "pure Python"
        }
    except ImportError as e:
        return {"module": module_name, "importable": False, "error": str(e)}

# Check common dependencies
deps_to_audit = ["json", "threading", "asyncio", "random", "statistics"]
print("C-extension audit:")
for dep in deps_to_audit:
    result = audit_extension(dep)
    status = "ok" if result.get("importable") else "MISSING"
    note = result.get('note', result.get('error', ''))
    print(f"  {dep:<15} {status:<8} {note}")

print()
print("For production: also check numpy, scipy, pandas, pydantic, etc.")
print("After all imports: sys._is_gil_enabled() must return False")

C-extension audit:
  json            ok       pure Python
  threading       ok       pure Python
  asyncio         ok       pure Python
  random          ok       pure Python
  statistics      ok       pure Python

For production: also check numpy, scipy, pandas, pydantic, etc.
After all imports: sys._is_gil_enabled() must return False


In [17]:
import os
import sys

# Gate 3: Feature-flag free-threading via environment variable
# In production: PYTHON_GIL=0 enables free-threading; PYTHON_GIL=1 re-enables GIL

def get_pool_mode() -> str:
    """Choose worker pool based on runtime environment."""
    # Check if explicitly requesting standard GIL behavior
    if os.environ.get('PYTHON_GIL') == '1':
        return 'interpreters'  # use sub-interpreters for isolation

    # On free-threaded build with GIL off: use thread pool
    if hasattr(sys, '_is_gil_enabled') and not sys._is_gil_enabled():
        return 'threads'

    # Standard build: use sub-interpreters or processes
    return 'interpreters'

mode = get_pool_mode()
print(f"Selected pool mode: {mode}")
print(f"GIL status: {getattr(sys, '_is_gil_enabled', lambda: True)()}")
print()
print("To toggle (in shell before starting service):")
print("  PYTHON_GIL=0  python3.14t service.py   # free threads")
print("  PYTHON_GIL=1  python3.14t service.py   # re-enable GIL, instant rollback")

Selected pool mode: interpreters
GIL status: True

To toggle (in shell before starting service):
  PYTHON_GIL=0  python3.14t service.py   # free threads
  PYTHON_GIL=1  python3.14t service.py   # re-enable GIL, instant rollback


### Closing the loop: final benchmark

The course ends with this result table for the Globomantics Risk Engine:

| Strategy | Build | Wall clock | Cores used | var95 |
|----------|-------|-----------|------------|-------|
| Single thread (baseline) | standard | ~1.41s | 1.0 | 71,618.80 |
| GIL threads (8) | standard | ~1.41s | 1.0 | 71,618.80 |
| Process pool (8) | standard | ~0.50s | 2.9 | 71,618.80 |
| Sub-interpreters (8) | standard | ~0.46s | ~7.5 | 71,618.80 |
| Free threads (8) | **3.14t** | ~0.38s | ~7.5 | 71,618.80 |

Key takeaways from this table:
- All strategies produce identical `var95` digits — correctness confirmed
- GIL threads show 1.0 cores-used → taking turns
- Sub-interpreters and free threads both achieve ~7.5 cores-used → genuine parallelism
- **"The lock didn't get faster, it got optional"**

---
<a id='decision-guide'></a>
## Decision Guide: Choosing Your Parallelism Strategy

```
What is your dominant workload?
│
├─ I/O bound (network, files, DB queries)
│   └─▶ asyncio is your front door — stay there
│       Add thread pool via asyncio.to_thread() if CPU pops up
│
└─ CPU bound (simulation, ML, image processing)
    │
    ├─ Can you use python3.14t in production?
    │   ├─ Yes, all deps have cp314t wheels
    │   │   └─▶ Free-threaded threads
    │   │       • Zero-copy handoffs, lowest latency
    │   │       • Shared heap → discipline required
    │   │       • ~20% single-thread overhead
    │   │
    │   └─ No, must stay on standard build
    │       └─▶ Sub-interpreters
    │           • No build change needed
    │           • Safety by construction (no shared state)
    │           • Crossing cost (pickle or shareable types)
    │
    └─ Jobs are untrusted, stateful, or fault-prone?
        └─▶ Sub-interpreters regardless of build
            • A corrupted interpreter dies alone
            • A racing thread corrupts the whole process
```

### Secondary decision criteria

| Concern | Prefer |
|---------|--------|
| Lowest latency, hot shared data | Free-threading with `functools.partial` |
| Untrusted or stateful jobs | Sub-interpreters |
| Standard-build-only constraint | Sub-interpreters |
| IO fan-out (web scraping, API calls) | asyncio stays the front door |
| Can tolerate blast radius of a crash | Sub-interpreters (isolated death) |

### Combining both

A free-threaded process can also host sub-interpreters:
- Use sub-interpreters for jobs that demand isolation (untrusted user code)
- Use free threads for jobs where shared data is hot and trusted
- **One production API, one `--pool` flag** selects the strategy per deployment

In [18]:
# Full summary: side-by-side properties of all four strategies

strategies = [
    {
        "name": "asyncio",
        "model": "cooperative, single thread",
        "parallel": False,
        "best_for": "IO-bound fan-out",
        "cost": "zero spawn, minimal memory",
        "safety": "single thread, no races",
        "build": "any",
    },
    {
        "name": "GIL threads",
        "model": "preemptive, one GIL",
        "parallel": False,  # for CPU-bound
        "best_for": "IO-bound concurrency",
        "cost": "cheapest, ~50µs per thread",
        "safety": "GIL serializes bytecode, not data",
        "build": "standard 3.14",
    },
    {
        "name": "Processes",
        "model": "separate processes, separate GILs",
        "parallel": True,
        "best_for": "CPU-bound, max isolation",
        "cost": "0.3s spawn, pickle on every call",
        "safety": "full isolation, crash-safe",
        "build": "any",
    },
    {
        "name": "Sub-interpreters",
        "model": "one process, per-interpreter GIL",
        "parallel": True,
        "best_for": "CPU-bound on standard build, untrusted jobs",
        "cost": "per-worker imports, pickle most values",
        "safety": "isolation by construction",
        "build": "standard 3.14 (no build swap)",
    },
    {
        "name": "Free threads",
        "model": "one process, no GIL",
        "parallel": True,
        "best_for": "CPU-bound, hot shared data",
        "cost": "~20% single-thread overhead, zero-copy handoffs",
        "safety": "discipline required (locks, partials)",
        "build": "python3.14t only",
    },
]

print(f"{'Strategy':<20} {'CPU parallel':>12} {'Build':<20} {'Best for'}")
print("-" * 80)
for s in strategies:
    parallel_str = "yes" if s["parallel"] else "no (IO)"
    print(f"{s['name']:<20} {parallel_str:>12} {s['build']:<20} {s['best_for']}")

Strategy             CPU parallel Build                Best for
--------------------------------------------------------------------------------
asyncio                   no (IO) any                  IO-bound fan-out
GIL threads               no (IO) standard 3.14        IO-bound concurrency
Processes                     yes any                  CPU-bound, max isolation
Sub-interpreters              yes standard 3.14 (no build swap) CPU-bound on standard build, untrusted jobs
Free threads                  yes python3.14t only     CPU-bound, hot shared data


---
## Quick Reference

### Key APIs

```python
# Check GIL status at runtime
import sys
sys._is_gil_enabled()  # False on 3.14t with no incompatible imports

# asyncio handoff to thread pool
result = await asyncio.to_thread(cpu_function, args)

# Sub-interpreter pool (Python 3.14+)
from concurrent.futures import InterpreterPoolExecutor
with InterpreterPoolExecutor(max_workers=N) as pool:
    results = list(pool.map(worker_fn, inputs))

# Feature flag: re-enable GIL on free-threaded build (instant rollback)
# -X gil=1  OR  PYTHON_GIL=1 python3.14t service.py

# Install free-threaded Python
# uv python install 3.14t
```

### Thread-safety checklist

Before deploying free-threaded:
- [ ] No unprotected shared mutable globals
- [ ] No lazy-init cache without a lock (check-then-act)
- [ ] No compound `+=` on shared counters without a lock
- [ ] No invariants spanning multiple fields without protecting both together
- [ ] All C extension deps have `cp314t` wheels or `Py_mod_gil` support
- [ ] CI runs on both `python3.14` and `python3.14t`
- [ ] Startup probe verifies `sys._is_gil_enabled() == False`

### Benchmarking checklist

Before publishing numbers:
- [ ] Fixed, representative workload (same N across all strategies)
- [ ] Warmed up before timing
- [ ] Reported best-of-N, not single sample
- [ ] Reported wall-clock AND cores-used
- [ ] Benchmarked against the build you would actually ship
- [ ] Showed both denominators when comparing builds
- [ ] Did not quote profiler-inflated seconds as performance numbers